# Experiment 5: Preprocessing & Normalization Audit

This notebook validates the jointly-normalized expression matrices for TCGA-BRCA, TCGA-OV, and I-SPY2.

**Outputs verified:**
- `tcga_brca_vst.parquet` / `tcga_ov_vst.parquet` -- VST-normalized TCGA expression
- `tcga_brca_ranks.parquet` / `tcga_ov_ranks.parquet` -- Rank-transformed TCGA
- `ispy2_normalized.parquet` / `ispy2_ranks.parquet` -- I-SPY2 expression
- `common_genes.txt` -- Cross-platform gene intersection
- `tcga_ov_hrd_scores.parquet` -- OV HRD labels

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

OUT = Path('/home/dani/repos2/Multiscale_HRD_Classifier/v2/experiments/exp5_normalization')
BASE = Path('/home/dani/repos2/Multiscale_HRD_Classifier')

print('Setup complete.')

Setup complete.


## 1. Load All Matrices

In [2]:
# Load all matrices
brca_vst = pd.read_parquet(OUT / 'tcga_brca_vst.parquet')
ov_vst = pd.read_parquet(OUT / 'tcga_ov_vst.parquet')
brca_ranks = pd.read_parquet(OUT / 'tcga_brca_ranks.parquet')
ov_ranks = pd.read_parquet(OUT / 'tcga_ov_ranks.parquet')
ispy2 = pd.read_parquet(OUT / 'ispy2_normalized.parquet')
ispy2_ranks = pd.read_parquet(OUT / 'ispy2_ranks.parquet')

with open(OUT / 'common_genes.txt') as f:
    common_genes = [l.strip() for l in f if l.strip()]

ov_hrd = pd.read_parquet(OUT / 'tcga_ov_hrd_scores.parquet')
ov_clin = pd.read_parquet(OUT / 'tcga_ov_clinical.parquet')

print(f'BRCA VST: {brca_vst.shape}')
print(f'OV VST: {ov_vst.shape}')
print(f'BRCA ranks: {brca_ranks.shape}')
print(f'OV ranks: {ov_ranks.shape}')
print(f'I-SPY2: {ispy2.shape}')
print(f'I-SPY2 ranks: {ispy2_ranks.shape}')
print(f'Common genes: {len(common_genes)}')
print(f'OV HRD scores: {len(ov_hrd)}')
print(f'OV clinical: {len(ov_clin)}')

BRCA VST: (1095, 18760)
OV VST: (426, 18760)
BRCA ranks: (1095, 18760)
OV ranks: (426, 18760)
I-SPY2: (21480, 105)
I-SPY2 ranks: (105, 21480)
Common genes: 17026
OV HRD scores: 177
OV clinical: 608


## 2. Distribution Checks

In [3]:
# Overall distribution of VST values
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# BRCA
sample_vals = brca_vst.iloc[:5].values.flatten()
axes[0].hist(sample_vals[np.isfinite(sample_vals)], bins=100, alpha=0.7, color='#2196F3')
axes[0].set_title(f'BRCA VST (5 samples)\nmedian={np.nanmedian(sample_vals):.1f}')
axes[0].set_xlabel('VST value')

# OV
sample_vals = ov_vst.iloc[:5].values.flatten()
axes[1].hist(sample_vals[np.isfinite(sample_vals)], bins=100, alpha=0.7, color='#4CAF50')
axes[1].set_title(f'OV VST (5 samples)\nmedian={np.nanmedian(sample_vals):.1f}')
axes[1].set_xlabel('VST value')

# I-SPY2
sample_vals = ispy2.iloc[:, :5].values.flatten()
axes[2].hist(sample_vals[np.isfinite(sample_vals)], bins=100, alpha=0.7, color='#FF9800')
axes[2].set_title(f'I-SPY2 log2 (5 samples)\nmedian={np.nanmedian(sample_vals):.1f}')
axes[2].set_xlabel('log2 expression')

plt.tight_layout()
plt.savefig(OUT / 'audit_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Key Gene Spot Checks

In [4]:
check_genes = ['BRCA1', 'BRCA2', 'POLQ', 'ESR1', 'TP53', 'MYC', 'CDK1', 'RAD51']
available_genes = [g for g in check_genes if g in brca_vst.columns]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for ax, gene in zip(axes.flat, available_genes):
    b = brca_vst[gene].values
    o = ov_vst[gene].values
    
    ax.hist(b, bins=50, alpha=0.6, label=f'BRCA (n={len(b)})', color='#2196F3', density=True)
    ax.hist(o, bins=50, alpha=0.6, label=f'OV (n={len(o)})', color='#4CAF50', density=True)
    
    if gene in ispy2.index:
        i_vals = ispy2.loc[gene].values
        ax2 = ax.twinx()
        ax2.hist(i_vals, bins=30, alpha=0.4, label=f'I-SPY2', color='#FF9800', density=True)
        ax2.set_ylabel('I-SPY2 density', fontsize=7, color='#FF9800')
    
    ax.set_title(gene, fontsize=11, fontweight='bold')
    ax.legend(fontsize=7)

plt.suptitle('VST Expression Distributions for Key Genes', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUT / 'audit_gene_spotcheck.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. BRCA vs OV Comparison (Joint Normalization Validation)

In [5]:
# Check that joint normalization doesn't create batch effects
# PCA on combined BRCA + OV
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

combined_vst = pd.concat([brca_vst, ov_vst], axis=0)
labels = ['BRCA'] * len(brca_vst) + ['OV'] * len(ov_vst)

# Subsample common genes for speed
common_in_tcga = [g for g in common_genes if g in combined_vst.columns]
X = combined_vst[common_in_tcga[:5000]].fillna(0).values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=5, random_state=42)
pc = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PC1 vs PC2
for label, color in [('BRCA', '#2196F3'), ('OV', '#4CAF50')]:
    mask = np.array(labels) == label
    axes[0].scatter(pc[mask, 0], pc[mask, 1], alpha=0.3, s=10, c=color, label=label)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
axes[0].set_title('PCA: BRCA vs OV (Joint VST)')
axes[0].legend()

# Explained variance
axes[1].bar(range(1, 6), pca.explained_variance_ratio_ * 100)
axes[1].set_xlabel('PC')
axes[1].set_ylabel('Variance Explained (%)')
axes[1].set_title('PCA Variance Explained')

plt.tight_layout()
plt.savefig(OUT / 'audit_pca_brca_ov.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'PC1 explains {pca.explained_variance_ratio_[0]:.1%} variance')
print(f'Total top-5 PCs: {sum(pca.explained_variance_ratio_):.1%}')

PC1 explains 14.3% variance
Total top-5 PCs: 34.2%


## 5. Rank Transformation Validation

In [6]:
# Verify ranks are uniform [0, 1]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(brca_ranks.iloc[0].values, bins=50, alpha=0.7, color='#2196F3')
axes[0].set_title(f'BRCA sample 1 ranks\nmin={brca_ranks.iloc[0].min():.3f}, max={brca_ranks.iloc[0].max():.3f}')

axes[1].hist(ov_ranks.iloc[0].values, bins=50, alpha=0.7, color='#4CAF50')
axes[1].set_title(f'OV sample 1 ranks\nmin={ov_ranks.iloc[0].min():.3f}, max={ov_ranks.iloc[0].max():.3f}')

axes[2].hist(ispy2_ranks.iloc[0].values, bins=50, alpha=0.7, color='#FF9800')
axes[2].set_title(f'I-SPY2 sample 1 ranks\nmin={ispy2_ranks.iloc[0].min():.3f}, max={ispy2_ranks.iloc[0].max():.3f}')

for ax in axes:
    ax.set_xlabel('Normalized rank [0, 1]')

plt.suptitle('Within-Sample Rank Distributions (should be ~uniform)', fontsize=12)
plt.tight_layout()
plt.savefig(OUT / 'audit_rank_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Cross-Platform Gene Rank Correlation

In [7]:
# Compare rank of BRCA1 across platforms
from scipy import stats

common_set = set(common_genes)
check_genes = [g for g in ['BRCA1', 'POLQ', 'ESR1', 'TP53', 'MYC', 'CDK1', 'AURKA', 'FOXM1'] 
               if g in common_set and g in brca_vst.columns and g in ispy2.index]

# Median rank of each gene across samples
brca_median_rank = brca_ranks.median(axis=0)
ov_median_rank = ov_ranks.median(axis=0)
ispy2_median_rank = ispy2_ranks.median(axis=0)

# Scatter of median gene ranks: BRCA vs OV
shared = [g for g in brca_median_rank.index if g in ov_median_rank.index]
r_brca_ov = stats.spearmanr(brca_median_rank[shared], ov_median_rank[shared])

# BRCA (RNA-seq) vs I-SPY2 (microarray) via rank
shared_ispy2 = [g for g in common_genes if g in brca_median_rank.index and g in ispy2_median_rank.index]

# I-SPY2 ranks are samples x genes (columns = genes)
ispy2_gene_median = ispy2_median_rank
r_brca_ispy2 = stats.spearmanr(brca_median_rank[shared_ispy2], ispy2_gene_median[shared_ispy2])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(brca_median_rank[shared], ov_median_rank[shared], alpha=0.1, s=3, color='gray')
for g in check_genes:
    if g in shared:
        axes[0].annotate(g, (brca_median_rank[g], ov_median_rank[g]), fontsize=8, color='red')
axes[0].set_xlabel('BRCA median rank')
axes[0].set_ylabel('OV median rank')
axes[0].set_title(f'Gene Rank Correlation: BRCA vs OV\nSpearman r = {r_brca_ov.statistic:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)

axes[1].scatter(brca_median_rank[shared_ispy2], ispy2_gene_median[shared_ispy2], alpha=0.1, s=3, color='gray')
for g in check_genes:
    if g in shared_ispy2:
        axes[1].annotate(g, (brca_median_rank[g], ispy2_gene_median[g]), fontsize=8, color='red')
axes[1].set_xlabel('BRCA median rank (RNA-seq)')
axes[1].set_ylabel('I-SPY2 median rank (microarray)')
axes[1].set_title(f'Cross-Platform Rank Correlation\nSpearman r = {r_brca_ispy2.statistic:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)

plt.tight_layout()
plt.savefig(OUT / 'audit_cross_platform_ranks.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'BRCA vs OV gene rank Spearman r = {r_brca_ov.statistic:.4f} (p = {r_brca_ov.pvalue:.2e})')
print(f'BRCA vs I-SPY2 gene rank Spearman r = {r_brca_ispy2.statistic:.4f} (p = {r_brca_ispy2.pvalue:.2e})')

BRCA vs OV gene rank Spearman r = 0.9430 (p = 0.00e+00)
BRCA vs I-SPY2 gene rank Spearman r = 0.5528 (p = 0.00e+00)


## 7. OV HRD Score Distribution

In [8]:
# Load BRCA HRD scores for comparison
brca_hrd = pd.read_excel(BASE / 'data/tcga.hrdscore.xlsx')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(brca_hrd['HRD-sum'].dropna(), bins=50, alpha=0.7, color='#2196F3', label='BRCA')
axes[0].hist(ov_hrd['HRD_Score'].dropna(), bins=50, alpha=0.7, color='#4CAF50', label='OV')
axes[0].axvline(42, color='red', ls='--', label='HRD threshold (42)')
axes[0].set_xlabel('HRD Score (sum of LOH + TAI + LST)')
axes[0].set_ylabel('Count')
axes[0].set_title('HRD Score Distribution')
axes[0].legend()

# OV: breakdown by component
for col, color in [('HRD_TAI', '#2196F3'), ('HRD_LST', '#4CAF50'), ('HRD_LOH', '#FF9800')]:
    vals = ov_hrd[col].dropna()
    axes[1].hist(vals, bins=30, alpha=0.5, color=color, label=f'{col} (mean={vals.mean():.1f})')
axes[1].set_xlabel('Score')
axes[1].set_ylabel('Count')
axes[1].set_title('OV HRD Components')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUT / 'audit_hrd_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary stats
brca_pos = (brca_hrd['HRD-sum'] >= 42).sum()
brca_neg = (brca_hrd['HRD-sum'] < 42).sum()
ov_pos = (ov_hrd['HRD_Score'] >= 42).sum()
ov_neg = (ov_hrd['HRD_Score'] < 42).sum()

print(f'BRCA: {brca_pos} HRD+ ({100*brca_pos/(brca_pos+brca_neg):.1f}%), {brca_neg} HRD-')
print(f'OV: {ov_pos} HRD+ ({100*ov_pos/(ov_pos+ov_neg):.1f}%), {ov_neg} HRD-')
print(f'OV patients with RNA-seq and HRD: {len(set(ov_hrd["patient_barcode"]) & set(ov_vst.index))}')

BRCA: 181 HRD+ (18.9%), 777 HRD-
OV: 96 HRD+ (55.5%), 77 HRD-
OV patients with RNA-seq and HRD: 177


## 8. Feature Survival Summary

In [9]:
# Load signature gene lists
consensus_df = pd.read_csv(BASE / 'v2/signature_analysis/consensus_hrd_genes.csv')
allsig_df = pd.read_csv(BASE / 'v2/signature_analysis/all_signature_gene_lists.csv')
dd500 = pd.read_csv(OUT / 'dd500_genes.csv')

common_set = set(common_genes)
tcga_genes = set(brca_vst.columns)

gene_sets = {
    'Consensus-93': set(consensus_df['Gene']),
    'All-Signature-410': set(allsig_df['Gene'].unique()),
    'DD-100': set(dd500.head(100)['gene']),
    'DD-500': set(dd500['gene']),
}

print('Feature Survival: TCGA (18,760 genes) -> Common (17,026 genes)')
print('=' * 70)
for name, genes in gene_sets.items():
    in_tcga = len(genes & tcga_genes)
    in_common = len(genes & common_set)
    print(f'  {name:>20s}: {len(genes):>3d} total -> {in_tcga:>3d} in TCGA -> {in_common:>3d} cross-platform ({100*in_common/len(genes):.1f}%)')

# Missing genes
print(f'\nMissing consensus genes (not in common set):')
missing = gene_sets['Consensus-93'] - common_set
for g in sorted(missing):
    in_tcga = g in tcga_genes
    in_ispy2 = g in set(ispy2.index)
    print(f'  {g}: in TCGA={in_tcga}, in I-SPY2={in_ispy2}')

Feature Survival: TCGA (18,760 genes) -> Common (17,026 genes)
          Consensus-93:  93 total ->  93 in TCGA ->  91 cross-platform (97.8%)
     All-Signature-410: 410 total -> 393 in TCGA -> 373 cross-platform (91.0%)
                DD-100: 100 total -> 100 in TCGA ->  89 cross-platform (89.0%)
                DD-500: 500 total -> 500 in TCGA -> 463 cross-platform (92.6%)

Missing consensus genes (not in common set):
  H2AX: in TCGA=True, in I-SPY2=False
  MRE11: in TCGA=True, in I-SPY2=False


## 9. Sample-Level QC

In [10]:
# Check for outlier samples (based on median expression)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

brca_medians = brca_vst.median(axis=1)
ov_medians = ov_vst.median(axis=1)
ispy2_medians = ispy2.median(axis=0)  # genes x samples, so median along axis=0

axes[0].hist(brca_medians, bins=50, color='#2196F3', alpha=0.7)
axes[0].set_title(f'BRCA per-sample median VST\nmean={brca_medians.mean():.2f}')
axes[0].axvline(brca_medians.mean(), color='red', ls='--')

axes[1].hist(ov_medians, bins=50, color='#4CAF50', alpha=0.7)
axes[1].set_title(f'OV per-sample median VST\nmean={ov_medians.mean():.2f}')
axes[1].axvline(ov_medians.mean(), color='red', ls='--')

axes[2].hist(ispy2_medians, bins=30, color='#FF9800', alpha=0.7)
axes[2].set_title(f'I-SPY2 per-sample median log2\nmean={ispy2_medians.mean():.2f}')
axes[2].axvline(ispy2_medians.mean(), color='red', ls='--')

plt.tight_layout()
plt.savefig(OUT / 'audit_sample_qc.png', dpi=150, bbox_inches='tight')
plt.show()

# Check for extreme outliers
for name, medians in [('BRCA', brca_medians), ('OV', ov_medians), ('I-SPY2', ispy2_medians)]:
    z = (medians - medians.mean()) / medians.std()
    outliers = (z.abs() > 3).sum()
    print(f'{name}: {outliers} outlier samples (|z| > 3)')

BRCA: 7 outlier samples (|z| > 3)
OV: 1 outlier samples (|z| > 3)
I-SPY2: 0 outlier samples (|z| > 3)


## 10. Summary Table

In [11]:
summary = pd.DataFrame([
    {'Dataset': 'TCGA-BRCA', 'Platform': 'RNA-seq', 'Normalization': 'Joint VST',
     'Samples': brca_vst.shape[0], 'Genes': brca_vst.shape[1],
     'NaN': brca_vst.isna().sum().sum(), 'Value Range': f'{brca_vst.values.min():.1f} - {brca_vst.values.max():.1f}'},
    {'Dataset': 'TCGA-OV', 'Platform': 'RNA-seq', 'Normalization': 'Joint VST',
     'Samples': ov_vst.shape[0], 'Genes': ov_vst.shape[1],
     'NaN': ov_vst.isna().sum().sum(), 'Value Range': f'{ov_vst.values.min():.1f} - {ov_vst.values.max():.1f}'},
    {'Dataset': 'I-SPY2', 'Platform': 'Agilent microarray', 'Normalization': 'RMA/quantile (original)',
     'Samples': ispy2.shape[1], 'Genes': ispy2.shape[0],
     'NaN': int(ispy2.isna().sum().sum()), 'Value Range': f'{ispy2.values[np.isfinite(ispy2.values)].min():.1f} - {ispy2.values[np.isfinite(ispy2.values)].max():.1f}'},
])

print('=== Experiment 5: Normalization Summary ===')
print(summary.to_string(index=False))
print(f'\nCommon genes across all platforms: {len(common_genes)}')
print(f'Consensus-93 survival: 91/93 (97.8%)')
print(f'DD-500 survival: 463/500 (92.6%)')
print(f'\nOV HRD scores available: {ov_hrd["HRD_Score"].notna().sum()} / {len(ov_hrd)}')
print(f'OV patients with RNA-seq + HRD: {len(set(ov_hrd["patient_barcode"]) & set(ov_vst.index))}')
print('\nAll outputs saved to: v2/experiments/exp5_normalization/')

=== Experiment 5: Normalization Summary ===
  Dataset           Platform           Normalization  Samples  Genes  NaN Value Range
TCGA-BRCA            RNA-seq               Joint VST     1095  18760    0  4.2 - 23.2
  TCGA-OV            RNA-seq               Joint VST      426  18760    0  4.2 - 21.9
   I-SPY2 Agilent microarray RMA/quantile (original)      105  21480   84  3.4 - 19.3

Common genes across all platforms: 17026
Consensus-93 survival: 91/93 (97.8%)
DD-500 survival: 463/500 (92.6%)

OV HRD scores available: 173 / 177
OV patients with RNA-seq + HRD: 177

All outputs saved to: v2/experiments/exp5_normalization/
